In [10]:
# Import Basic Packgaes 
import numpy as np
import pandas as pd
from datetime import datetime
import statsmodels as sm
import itertools
import nptdms
import glob
import mysql.connector
import sqlalchemy

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns # advanced vizs
from pandas.plotting import lag_plot

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Set Color Palettes
palette1 = itertools.cycle(sns.color_palette(palette='Set1'))
palette2 = itertools.cycle(sns.color_palette(palette='Set2'))

In [26]:
# Define the path to your TDMS files
tdms_path = "C:\\Users\\11035628\\OneDrive - Dover Corporation\\AODD Pump Project\\P-1,28Apr23,Data Plot\\*.tdms"

# Get a list of file paths matching the pattern
file_list = glob.glob(tdms_path)

In [27]:
# Loop over the files and read them into pandas data frames
dfs = []
for file in file_list:
    with nptdms.TdmsFile.read(file) as tdms_file:
        groups = tdms_file.groups()
        for group in groups:
            data = group.as_dataframe()
            dfs.append(data)
            
# Concatenate the data frames into a single data frame
pump_1D_df = pd.concat(dfs)

In [28]:
# Combine the time data columns into a string column
pump_1D_df['time_str'] = (pump_1D_df.apply(lambda row: f"{row['Time(Hrs)']:02}:{row['Time(Min)']:02}:{row['Time(Sec)']:02}", axis=1))
#Insert Date Column
pump_1D_df.insert(18,'Date','2023/04/28') # insert correct date
#Combine Date and Time Column
pump_1D_df['Correct_Time'] = pump_1D_df['Date'] +' '+ pump_1D_df['time_str']

In [ ]:
pump_1D_df.Correct_Time = pd.to_datetime(pump_1D_df.Correct_Time)

In [29]:
pump_1D_df.head()

,Time,Time(Hrs),Time(Min),Time(Sec),Cycle-detect,P1-CPM,P1-Cycle-Count,P1-Water Suction Pressure,P1-Channel-1,P1-Water Discharge Pressure,P1-Channel-3,P1-Air-Supply-pressure,P1-Water Suction Flowrate,P1-Water Discharge Flowrate,P1-Air Supply Flowrate,filename,group,time_str,Date,Correct_Time
0,2023-04-27 18:53:04.550059,0.0,20.0,38.27,0.0,50.470179,45898.0,4.282332,0.0,38.806727,0.0,105.840628,69.750380,73.137957,67.281793,C:\Users\11035628\OneDrive - Dover Corporation...,/'Untitled',0.0:20.0:38.27,2023/04/28,2023/04/28 0.0:20.0:38.27
1,2023-04-27 18:53:05.550059,0.0,20.0,38.30,0.0,50.470179,45898.0,6.892468,0.0,37.335764,0.0,106.050765,69.837241,73.224818,67.542454,C:\Users\11035628\OneDrive - Dover Corporation...,/'Untitled',0.0:20.0:38.3,2023/04/28,2023/04/28 0.0:20.0:38.3
2,2023-04-27 18:53:06.550059,0.0,20.0,38.33,0.0,50.470179,45898.0,7.564846,0.0,38.946819,0.0,105.840628,69.837241,73.137957,67.542454,C:\Users\11035628\OneDrive - Dover Corporation...,/'Untitled',0.0:20.0:38.33,2023/04/28,2023/04/28 0.0:20.0:38.33
3,2023-04-27 18:53:07.550059,0.0,20.0,38.36,0.0,50.470179,45898.0,9.423226,0.0,38.386452,0.0,106.120811,69.837241,73.137957,67.151463,C:\Users\11035628\OneDrive - Dover Corporation...,/'Untitled',0.0:20.0:38.36,2023/04/28,2023/04/28 0.0:20.0:38.36
4,2023-04-27 18:53:08.550059,0.0,20.0,38.39,0.0,50.470179,45898.0,10.968763,0.0,40.557874,0.0,106.050765,69.750380,73.051096,67.412123,C:\Users\11035628\OneDrive - Dover Corporation...,/'Untitled',0.0:20.0:38.39,2023/04/28,2023/04/28 0.0:20.0:38.39


In [30]:
pump_1D_df.shape

(2292533, 20)

In [31]:
pump_1D_df.columns

Index(['Time', 'Time(Hrs)', 'Time(Min)', 'Time(Sec)', 'Cycle-detect', 'P1-CPM',
       'P1-Cycle-Count', 'P1-Water Suction Pressure', 'P1-Channel-1',
       'P1-Water Discharge Pressure', 'P1-Channel-3', 'P1-Air-Supply-pressure',
       'P1-Water Suction Flowrate', 'P1-Water Discharge Flowrate',
       'P1-Air Supply Flowrate', 'filename', 'group', 'time_str', 'Date',
       'Correct_Time'],
      dtype='object')

In [32]:
# This has to be done otherwise data cannot be pushed to SQL server
inf_count = np.isinf(pump_1D_df['P1-CPM']).values.sum()
print(inf_count)

1334


In [33]:
pump_1D_df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [34]:
inf_count = np.isinf(pump_1D_df['P1-CPM']).values.sum()
print(inf_count)

0


In [38]:
database_username = 'root'
database_password = '****' # use your pwd
database_ip       = '127.0.0.1'
database_name     = 'p1_aodd_pump_raw_data'
database_connection = sqlalchemy.create_engine('mysql+mysqlconnector://{0}:{1}@{2}/{3}'.
                                               format(database_username, database_password, 
                                                      database_ip, database_name), pool_recycle=1, pool_timeout=57600).connect()

pump_1D_df.to_sql(con=database_connection, name='p1_lt_2023_04_28', if_exists='replace',chunksize=10000) # replace name with the corresponding date
database_connection.close()

In [8]:
pump_1D_df.drop(columns=['Time', 'Time(Hrs)', 'Time(Min)', 'Time(Sec)', 'Cycle-detect',
       'P1-Cycle-Count', 'P1-Channel-1',
       'P1-Channel-3', 'filename', 'group', 'time_str', 'Date',],inplace=True)
pump_1D_df.head()

,P1-CPM,P1-Water Suction Pressure,P1-Water Discharge Pressure,P1-Air-Supply-pressure,P1-Water Suction Flowrate,P1-Water Discharge Flowrate,P1-Air Supply Flowrate,Correct_Time
0,50.470179,4.282332,38.806727,105.840628,69.750380,73.137957,67.281793,2023/04/28 0.0:20.0:38.27
1,50.470179,6.892468,37.335764,106.050765,69.837241,73.224818,67.542454,2023/04/28 0.0:20.0:38.3
2,50.470179,7.564846,38.946819,105.840628,69.837241,73.137957,67.542454,2023/04/28 0.0:20.0:38.33
3,50.470179,9.423226,38.386452,106.120811,69.837241,73.137957,67.151463,2023/04/28 0.0:20.0:38.36
4,50.470179,10.968763,40.557874,106.050765,69.750380,73.051096,67.412123,2023/04/28 0.0:20.0:38.39


In [9]:
pump_1D_df['Time'] = pd.to_datetime(pump_1D_df['Correct_Time'], infer_datetime_format=True)
pump_1D_df.head()

,P1-CPM,P1-Water Suction Pressure,P1-Water Discharge Pressure,P1-Air-Supply-pressure,P1-Water Suction Flowrate,P1-Water Discharge Flowrate,P1-Air Supply Flowrate,Correct_Time,Time
0,50.470179,4.282332,38.806727,105.840628,69.750380,73.137957,67.281793,2023/04/28 0.0:20.0:38.27,2023-04-28 00:20:38.270
1,50.470179,6.892468,37.335764,106.050765,69.837241,73.224818,67.542454,2023/04/28 0.0:20.0:38.3,2023-04-28 00:20:38.300
2,50.470179,7.564846,38.946819,105.840628,69.837241,73.137957,67.542454,2023/04/28 0.0:20.0:38.33,2023-04-28 00:20:38.330
3,50.470179,9.423226,38.386452,106.120811,69.837241,73.137957,67.151463,2023/04/28 0.0:20.0:38.36,2023-04-28 00:20:38.360
4,50.470179,10.968763,40.557874,106.050765,69.750380,73.051096,67.412123,2023/04/28 0.0:20.0:38.39,2023-04-28 00:20:38.390


In [10]:
pump_1D_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2292533 entries, 0 to 50002
Data columns (total 9 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   P1-CPM                       float64       
 1   P1-Water Suction Pressure    float64       
 2   P1-Water Discharge Pressure  float64       
 3   P1-Air-Supply-pressure       float64       
 4   P1-Water Suction Flowrate    float64       
 5   P1-Water Discharge Flowrate  float64       
 6   P1-Air Supply Flowrate       float64       
 7   Correct_Time                 object        
 8   Time                         datetime64[ns]
dtypes: datetime64[ns](1), float64(7), object(1)
memory usage: 174.9+ MB


In [11]:
pump_1D_df.Time.duplicated().sum()

1815

In [12]:
pump_1D_df.drop_duplicates(subset=['Time'],keep='first',inplace=True)
pump_1D_df.Time.duplicated().sum()

0

In [13]:
pump_1D_df.drop(columns=['Correct_Time'],inplace=True)
pump_1D_df.head()

,P1-CPM,P1-Water Suction Pressure,P1-Water Discharge Pressure,P1-Air-Supply-pressure,P1-Water Suction Flowrate,P1-Water Discharge Flowrate,P1-Air Supply Flowrate,Time
0,50.470179,4.282332,38.806727,105.840628,69.750380,73.137957,67.281793,2023-04-28 00:20:38.270
1,50.470179,6.892468,37.335764,106.050765,69.837241,73.224818,67.542454,2023-04-28 00:20:38.300
2,50.470179,7.564846,38.946819,105.840628,69.837241,73.137957,67.542454,2023-04-28 00:20:38.330
3,50.470179,9.423226,38.386452,106.120811,69.837241,73.137957,67.151463,2023-04-28 00:20:38.360
4,50.470179,10.968763,40.557874,106.050765,69.750380,73.051096,67.412123,2023-04-28 00:20:38.390


In [14]:
pump_1D_df.Time = pd.to_datetime(pump_1D_df.Time)

In [15]:
pump_1D_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2290718 entries, 0 to 50002
Data columns (total 8 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   P1-CPM                       float64       
 1   P1-Water Suction Pressure    float64       
 2   P1-Water Discharge Pressure  float64       
 3   P1-Air-Supply-pressure       float64       
 4   P1-Water Suction Flowrate    float64       
 5   P1-Water Discharge Flowrate  float64       
 6   P1-Air Supply Flowrate       float64       
 7   Time                         datetime64[ns]
dtypes: datetime64[ns](1), float64(7)
memory usage: 157.3 MB


In [16]:
pump_1D_df.Time.duplicated().sum()

0

In [17]:
pump_1D_df.set_index(['Time'], inplace=True)
pump_1D_df.head()

,P1-CPM,P1-Water Suction Pressure,P1-Water Discharge Pressure,P1-Air-Supply-pressure,P1-Water Suction Flowrate,P1-Water Discharge Flowrate,P1-Air Supply Flowrate
Time,,,,,,,
2023-04-28 00:20:38.270,50.470179,4.282332,38.806727,105.840628,69.750380,73.137957,67.281793
2023-04-28 00:20:38.300,50.470179,6.892468,37.335764,106.050765,69.837241,73.224818,67.542454
2023-04-28 00:20:38.330,50.470179,7.564846,38.946819,105.840628,69.837241,73.137957,67.542454
2023-04-28 00:20:38.360,50.470179,9.423226,38.386452,106.120811,69.837241,73.137957,67.151463
2023-04-28 00:20:38.390,50.470179,10.968763,40.557874,106.050765,69.750380,73.051096,67.412123


In [18]:
pump_1D_df.duplicated().sum()

13715

In [19]:
pump_1D_df.drop_duplicates(keep='last',inplace=True)
pump_1D_df.duplicated().sum()

0

In [21]:
pump_1D_df.shape

(2277003, 7)

In [ ]:
database_username = 'root'
database_password = '****' # use your pwd
database_ip       = '127.0.0.1'
database_name     = 'p1_aodd_pump_cleaned_data' # create this database
database_connection = sqlalchemy.create_engine('mysql+mysqlconnector://{0}:{1}@{2}/{3}'.
                                               format(database_username, database_password, 
                                                      database_ip, database_name), pool_recycle=1, pool_timeout=57600).connect()

pump_1D_df.to_sql(con=database_connection, name='p1_lt_2023_04_28_clean', if_exists='replace',chunksize=10000) # replace name with the corresponding date
database_connection.close()

In [ ]:
plt.rcParams['figure.figsize'] = 15,10
pump_1D_df.plot(colormap='Paired')
plt.title('AODD Pump 1 Day Data', size=14)
plt.grid()